In [ ]:
import os
import torch
import random
import shutil
import pickle
import numpy as np
import torchvision
import torch.nn as nn

from tqdm import tqdm
from matplotlib import pyplot as plt

from torch.optim import Adam
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
config = {
    'model_params': {
        'in_channels': 1,
        'convbn_blocks': 3,
        'conv_kernel_size': [2, 2, 2],
        'conv_kernel_strides': [2, 2, 2],
        'convbn_channels': [1, 16, 32, 64],
        'enc_fc_layers': [576, 128, 2],
        'enc_fc_mu_activation': None,
        'enc_fc_var_activation': None,
        'conv_activation_fn': 'leaky',
        'dec_fc_layers': [2, 128, 576],
        'dec_fc_activation_fn': 'leaky',
        'enc_fc_activation_fn': 'leaky',
        'transpose_bn_blocks': 3,
        'transposebn_channels': [64, 32, 16, 1],
        'transpose_kernel_size': [3, 2, 2],
        'transpose_kernel_strides': [2, 2, 2],
        'transpose_activation_fn': 'tanh',
        'log_variance': True,
        'latent_dim': 2,
        'concat_channel': False,
        'decoder_fc_condition': False,
        'num_classes': 10,
        'conditional': False
    },
    'train_params': {
        'task_name': 'vae_kl',
        'batch_size': 64,
        'epochs': 10,
        'kl_weight': 1e-5,
        'lr': 0.005,
        'crit': 'l2',
        'ckpt_name': 'best_vae_kl_ckpt.pth',
        'seed': 111,
        'save_training_image': False,
        'output_train_dir': 'output',
        'save_latent_plot': True
    }
}


In [ ]:
import torch
import torch.nn as nn

r"""
A very simple VAE which has the following architecture

Encoder
    For Conditional model we stack num_classes empty channels onto the image
    We make the gt_label index channel as `1`

    N * Conv BN Activation Blocks
    FC layers for mean
    FC layers for variance

Decoder
    For Conditional model we also concat the one hot label feature onto the z input

    FC Layers taking z to higher dimensional feature
    N * ConvTranspose BN Activation Blocks
"""

class VAE(nn.Module):
    def __init__(self, config):
        super(VAE, self).__init__()

        activation_map = {
            'relu': nn.ReLU(),
            'leaky': nn.LeakyReLU(),
            'tanh': nn.Tanh(),
            'gelu': nn.GELU(),
            'silu': nn.SiLU()
        }

        self.config = config

        assert config['transpose_activation_fn'] is None or config['transpose_activation_fn'] in activation_map
        assert config['dec_fc_activation_fn'] is None or config['dec_fc_activation_fn'] in activation_map
        assert config['conv_activation_fn'] is None or config['conv_activation_fn'] in activation_map
        assert config['enc_fc_activation_fn'] is None or config['enc_fc_activation_fn'] in activation_map
        assert config['enc_fc_layers'][-1] == config['dec_fc_layers'][0] == config['latent_dim'], \
            "Latent dimension must be same as fc layers number"

        self.num_classes = config['num_classes']
        self.transposebn_channels = config['transposebn_channels']
        self.latent_dim = config['latent_dim']
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        model_config = dict(config)

        if model_config['concat_channel'] and model_config['conditional']:
            model_config['convbn_channels'] = model_config['convbn_channels'].copy()
            model_config['convbn_channels'][0] += self.num_classes

        self.encoder_layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    model_config['convbn_channels'][i],
                    model_config['convbn_channels'][i + 1],
                    kernel_size=model_config['conv_kernel_size'][i],
                    stride=model_config['conv_kernel_strides'][i]
                ),
                nn.BatchNorm2d(model_config['convbn_channels'][i + 1]),
                activation_map[model_config['conv_activation_fn']]
            )
            for i in range(model_config['convbn_blocks'])
        ])

        encoder_mu_activation = (
            nn.Identity()
            if model_config['enc_fc_mu_activation'] is None
            else activation_map[model_config['enc_fc_mu_activation']]
        )

        self.encoder_mu_fc = nn.ModuleList([
            nn.Sequential(
                nn.Linear(model_config['enc_fc_layers'][i], model_config['enc_fc_layers'][i + 1]),
                encoder_mu_activation
            )
            for i in range(len(model_config['enc_fc_layers']) - 1)
        ])

        encoder_var_activation = (
            nn.Identity()
            if model_config['enc_fc_var_activation'] is None
            else activation_map[model_config['enc_fc_var_activation']]
        )

        self.encoder_var_fc = nn.ModuleList([
            nn.Sequential(
                nn.Linear(model_config['enc_fc_layers'][i], model_config['enc_fc_layers'][i + 1]),
                encoder_var_activation
            )
            for i in range(len(model_config['enc_fc_layers']) - 1)
        ])

        if model_config['decoder_fc_condition'] and model_config['conditional']:
            model_config['dec_fc_layers'] = model_config['dec_fc_layers'].copy()
            model_config['dec_fc_layers'][0] += self.num_classes

        self.decoder_fc = nn.ModuleList([
            nn.Sequential(
                nn.Linear(model_config['dec_fc_layers'][i], model_config['dec_fc_layers'][i + 1]),
                activation_map[model_config['dec_fc_activation_fn']]
            )
            for i in range(len(model_config['dec_fc_layers']) - 1)
        ])

        self.decoder_layers = nn.ModuleList([
            nn.Sequential(
                nn.ConvTranspose2d(
                    model_config['transposebn_channels'][i],
                    model_config['transposebn_channels'][i + 1],
                    kernel_size=model_config['transpose_kernel_size'][i],
                    stride=model_config['transpose_kernel_strides'][i]
                ),
                nn.BatchNorm2d(model_config['transposebn_channels'][i + 1]),
                activation_map[model_config['transpose_activation_fn']]
            )
            for i in range(model_config['transpose_bn_blocks'])
        ])

    def forward(self, x, label=None):
        out = x

        if self.config['concat_channel'] and self.config['conditional']:
            label_ch_map = torch.zeros((x.size(0), self.num_classes, *x.shape[2:]), device=self.device)
            batch_idx = torch.arange(0, x.size(0), device=self.device)
            label_idx = label[batch_idx]
            label_ch_map[batch_idx, label_idx, :, :] = 1
            out = torch.cat([x, label_ch_map], dim=1)

        for layer in self.encoder_layers:
            out = layer(out)

        out = out.reshape((x.size(0), -1))

        mu = out
        for layer in self.encoder_mu_fc:
            mu = layer(mu)

        std = out
        for layer in self.encoder_var_fc:
            std = layer(std)

        z = self.reparameterize(mu, std)
        generated_out = self.generate(z, label)

        if self.config['log_variance']:
            return {
                'mean': mu,
                'log_variance': std,
                'image': generated_out,
            }
        else:
            return {
                'mean': mu,
                'std': std,
                'image': generated_out,
            }

    def generate(self, z, label=None):
        out = z

        if self.config['decoder_fc_condition'] and self.config['conditional']:
            assert label is not None, "Label cannot be none for conditional generation"

            label_fc_input = torch.zeros((z.size(0), self.num_classes), device=self.device)
            batch_idx = torch.arange(0, z.size(0), device=self.device)
            label_idx = label[batch_idx]
            label_fc_input[batch_idx, label_idx] = 1
            out = torch.cat([out, label_fc_input], dim=-1)

        for layer in self.decoder_fc:
            out = layer(out)

        hw = out.size(-1) / self.transposebn_channels[0]
        spatial = int(np.sqrt(hw))
        assert spatial * spatial == hw

        out = out.reshape((z.size(0), -1, spatial, spatial))

        for layer in self.decoder_layers:
            out = layer(out)

        return out

    def sample(self, label=None, num_images=1, z=None):
        if z is None:
            z = torch.randn((num_images, self.latent_dim), device=self.device)

        if self.config['conditional']:
            assert label is not None, "Label cannot be none for conditional sampling"
            assert label.size(0) == num_images

        assert z.size(0) == num_images
        out = self.generate(z, label)
        return out

    def reparameterize(self, mu, std_or_logvariance):
        if self.config['log_variance']:
            std = torch.exp(0.5 * std_or_logvariance)
        else:
            std = std_or_logvariance

        z = torch.randn_like(std)
        return z * std + mu




In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=config['train_params']['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['train_params']['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

In [ ]:
def train_for_one_epoch(epoch_idx, model, data_loader, optimizer, criterion, config):
    recon_losses = []
    kl_losses = []
    losses = []

    model.train()

    for im, label in tqdm(data_loader):
        im = im.float().to(device)
        label = label.long().to(device)

        optimizer.zero_grad()

        output = model(im, label)
        mean = output['mean']

        if config['model_params']['log_variance']:
            log_variance = output['log_variance']
            kl_loss = torch.mean(
                0.5 * torch.sum(torch.exp(log_variance) + mean ** 2 - 1 - log_variance, dim=-1)
            )
        else:
            std = output['std']
            kl_loss = torch.mean(
                0.5 * torch.sum(std ** 2 + mean ** 2 - 1 - torch.log(std ** 2 + 1e-8), dim=-1)
            )

        generated_im = output['image']
        recon_loss = criterion(generated_im, im)
        loss = recon_loss + config['train_params']['kl_weight'] * kl_loss

        recon_losses.append(recon_loss.item())
        kl_losses.append(kl_loss.item())
        losses.append(loss.item())

        loss.backward()
        optimizer.step()

    print(
        f'Epoch {epoch_idx+1} | Recon Loss: {np.mean(recon_losses):.4f} | KL Loss: {np.mean(kl_losses):.4f}'
    )

    return np.mean(losses)


In [ ]:
def reconstruct(config, model, dataset, num_images=100):
    print('Generating reconstructions')

    save_dir = os.path.join(config['train_params']['task_name'], config['train_params']['output_train_dir'])
    os.makedirs(save_dir, exist_ok=True)

    idxs = torch.randint(0, len(dataset), (num_images,))
    ims = torch.stack([dataset[idx][0] for idx in idxs]).float().to(device)
    labels = torch.tensor([dataset[idx][1] for idx in idxs], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(ims, labels)
        generated_im = output['image']

    ims = (ims + 1) / 2
    generated_im = (generated_im + 1) / 2

    out = torch.cat([ims, generated_im], dim=3)
    grid = make_grid(out, nrow=10)

    img = torchvision.transforms.ToPILImage()(grid.cpu())
    img.save(os.path.join(save_dir, 'reconstruction.png'))
    plt.figure(figsize=(12, 12))
    plt.imshow(np.array(img), cmap='gray')
    plt.axis('off')
    plt.show()


def visualize_latent_space(config, model, data_loader, save_fig_path):
    labels = []
    means = []

    model.eval()
    with torch.no_grad():
        for im, label in tqdm(data_loader):
            im = im.float().to(device)
            label = label.long().to(device)
            output = model(im, label)
            labels.append(label.cpu())
            means.append(output['mean'].cpu())

    labels = torch.cat(labels, dim=0).reshape(-1)
    means = torch.cat(means, dim=0)

    if model.latent_dim != 2:
        print('Latent dimension > 2 and hence projecting')
        U, S, V = torch.pca_lowrank(means, center=True, niter=2)
        means = torch.matmul(means, V[:, :2])

        os.makedirs(config['train_params']['task_name'], exist_ok=True)
        pickle.dump(V, open(f"{config['train_params']['task_name']}/pca_matrix.pkl", 'wb'))

    plt.figure(figsize=(8, 6))
    for num in range(10):
        idxs = torch.where(labels == num)[0]
        plt.scatter(
            means[idxs, 0].numpy(),
            means[idxs, 1].numpy(),
            s=8,
            label=str(num),
            alpha=0.7
        )

    plt.legend()
    plt.grid(True)
    plt.savefig(save_fig_path)
    plt.show()


def visualize_interpolation(config, model, dataset, interpolation_steps=20, save_dir='interp'):
    print('Interpolating between images')

    base_dir = os.path.join(config['train_params']['task_name'], config['train_params']['output_train_dir'])
    os.makedirs(base_dir, exist_ok=True)

    interp_dir = os.path.join(base_dir, save_dir)
    if os.path.exists(interp_dir):
        shutil.rmtree(interp_dir)
    os.mkdir(interp_dir)

    idxs = torch.randint(0, len(dataset), (2,))
    ims = torch.stack([dataset[idx][0] for idx in idxs]).float().to(device)
    labels = torch.tensor([dataset[idx][1] for idx in idxs], dtype=torch.long).to(device)

    with torch.no_grad():
        means = model(ims, labels)['mean']

        factors = torch.linspace(0, 1.0, steps=interpolation_steps).to(device)
        means_start = means[0]
        means_end = means[1]
        interp_means = factors[:, None] * means_end[None, :] + (1 - factors[:, None]) * means_start[None, :]
        out = model.generate(interp_means)

    out = (out + 1) / 2
    grid = make_grid(out, nrow=interpolation_steps)
    img = torchvision.transforms.ToPILImage()(grid.cpu())
    img.save(os.path.join(base_dir, 'interpolation.png'))

    plt.figure(figsize=(20, 3))
    plt.imshow(np.array(img), cmap='gray')
    plt.axis('off')
    plt.show()


def visualize_manifold(config, model):
    if model.latent_dim != 2:
        print('Manifold visualization is only supported directly for latent_dim=2')
        return

    print('Generating manifold')

    save_dir = os.path.join(config['train_params']['task_name'], config['train_params']['output_train_dir'])
    os.makedirs(save_dir, exist_ok=True)

    xs = torch.linspace(-3, 3, 20)
    ys = torch.linspace(-3, 3, 20)
    zs = []

    for y in ys:
        for x in xs:
            zs.append(torch.tensor([x, y]))

    zs = torch.stack(zs).float().to(device)

    with torch.no_grad():
        generated_ims = model.sample(num_images=zs.size(0), z=zs)

    generated_ims = (generated_ims + 1) / 2
    grid = make_grid(generated_ims, nrow=20)
    img = torchvision.transforms.ToPILImage()(grid.cpu())
    img.save(os.path.join(save_dir, 'manifold.png'))

    plt.figure(figsize=(10, 10))
    plt.imshow(np.array(img), cmap='gray')
    plt.axis('off')
    plt.show()


In [ ]:
def get_model(config):
    model = VAE(config=config['model_params'])
    return model

In [ ]:
def train(config, train_loader, test_loader):
    print(config)

    seed = config['train_params']['seed']
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = get_model(config).to(device)

    optimizer = Adam(model.parameters(), lr=config['train_params']['lr'])
    scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=1, verbose=True)

    criterion = {
        'l1': torch.nn.L1Loss(),
        'l2': torch.nn.MSELoss()
    }[config['train_params']['crit']]

    task_dir = config['train_params']['task_name']
    output_dir = os.path.join(task_dir, config['train_params']['output_train_dir'])

    os.makedirs(output_dir, exist_ok=True)

    best_loss = np.inf
    latent_im_path = os.path.join(output_dir, 'latent_epoch_{}.png')

    visualize_latent_space(config, model, test_loader, save_fig_path=latent_im_path.format(0))

    for epoch_idx in range(config['train_params']['epochs']):
        mean_loss = train_for_one_epoch(epoch_idx, model, train_loader, optimizer, criterion, config)

        if config['train_params']['save_latent_plot']:
            visualize_latent_space(
                config,
                model,
                test_loader,
                save_fig_path=latent_im_path.format(epoch_idx + 1)
            )

        scheduler.step(mean_loss)

        if mean_loss < best_loss:
            print(f'Improved Loss to {mean_loss:.4f} .... Saving Model')
            torch.save(
                model.state_dict(),
                os.path.join(task_dir, config['train_params']['ckpt_name'])
            )
            best_loss = mean_loss
        else:
            print('No Loss Improvement')

    return model


In [ ]:
def inference(config, test_dataset, test_loader):
    model = get_model(config).to(device)
    model.load_state_dict(
        torch.load(
            os.path.join(config['train_params']['task_name'], config['train_params']['ckpt_name']),
            map_location=device
        )
    )
    model.eval()

    latent_im_path = os.path.join(
        config['train_params']['task_name'],
        config['train_params']['output_train_dir'],
        'latent_inference.png'
    )

    visualize_latent_space(config, model, test_loader, latent_im_path)
    visualize_interpolation(config, model, test_dataset)
    reconstruct(config, model, test_dataset)
    visualize_manifold(config, model)


In [ ]:
model = train(config, train_loader, test_loader)


In [ ]:
inference(config, test_dataset, test_loader)
